#### Import Dependencies

In [1]:
import openai
import instructor
from qdrant_client import QdrantClient

from pydantic import BaseModel, Field

In [2]:
prompt = 'You are a helpful assistant. Return an answer to the question. Question: What is your name?'

In [3]:
response = openai.chat.completions.create(
    model='gpt-4.1-mini',
    messages=[{'role':'system', 'content': prompt}],
    temperature = 0
)

print(response.choices[0].message.content)

I am called ChatGPT. How can I assist you today?


#### Add Instructor (Structured Outputs)

In [4]:
client = instructor.from_openai(openai.OpenAI())

In [12]:
class RAGGenerationResponse(BaseModel):
    answer: str=Field(description='The answer to the question')

In [15]:
response, raw_response = client.chat.completions.create_with_completion(
    model='gpt-4.1-mini',
    messages=[{'role':'system', 'content': prompt}],
    temperature = 0,
    response_model=RAGGenerationResponse
)


In [14]:
response

RAGGenerationResponse(answer='I am ChatGPT, your AI assistant. How can I help you today?')

In [16]:
raw_response

ChatCompletion(id='chatcmpl-EnVYfmqmqAjpIO53Kpftyl3eqgNfV', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ORbGgg6faEE9qgePgeHAK98E', function=Function(arguments='{"answer":"My name is ChatGPT. How can I assist you today?"}', name='RAGGenerationResponse'), type='function')]))], created=1795468089, model='gpt-4.1-mini-2025-04-14', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_46411d0fe9', usage=CompletionUsage(completion_tokens=17, prompt_tokens=94, total_tokens=111, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0)))

In [17]:
class RAGGenerationResponse(BaseModel):
    answer: str=Field(description='The answer to the question')
    reasoning: str=Field(description='The reasoning for the answer')

In [18]:
response, raw_response = client.chat.completions.create_with_completion(
    model='gpt-4.1-mini',
    messages=[{'role':'system', 'content': prompt}],
    temperature = 0,
    response_model=RAGGenerationResponse
)


In [19]:
response

RAGGenerationResponse(answer='My name is ChatGPT.', reasoning='I am an AI language model created by OpenAI, and I am commonly referred to as ChatGPT.')

#### RAG Example

In [24]:

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding
#### Retrieval function

def retrieve_similar_books(query, qdrant_client, top_k=5):

    query_embedding = get_embedding(query)
    
    search_result = qdrant_client.query_points(
        collection_name="amazon_books_collection_sampled",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in search_result.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context.append(result.payload['description'])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload['average_rating'])
    
    return {
        "context_ids": retrieved_context_ids,
        "context": retrieved_context,
        "similarity_scores": similarity_scores,
        "ratings": retrieved_context_ratings
    }
    

def process_context(retrieved_context):
    formatted_context = ""

    for id, chunk, rating in zip(retrieved_context["context_ids"], retrieved_context["context"], retrieved_context["ratings"]):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n\n"
    return formatted_context
#### Create prompt function

def build_prompt(preprocessed_context, user_query):
    prompt = f"""
    You are a book recommendation assistant that can answer questions about the books in stock.

    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products
    
    Context:
    {preprocessed_context}
    
    Question: {user_query}
    
    Answer:
    """
    return prompt


def generate_answer(prompt):
    response,raw_response = client.chat.completions.create_with_completion(
        model = 'gpt-4.1-mini',
        messages =[{"role": "system", "content": prompt}],
        temperature = 0,
        response_model=RAGGenerationResponse
    )
    return response


def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url= "http://localhost:6333")

    retrieved_context = retrieve_similar_books(question, qdrant_client, top_k)
    processed_context = process_context(retrieved_context)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        'answer': answer.answer,
        'question': question,
        'retrieved_context': retrieved_context,
        'retrieved_context_ids': retrieved_context["context_ids"],
        'similarity_scores': retrieved_context["similarity_scores"]
    }
    return final_result

In [22]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [25]:
output = rag_pipeline('Good historical fiction about civil war')

In [26]:
output

{'answer': '"I Shall Be Near to You: A Novel" is a highly recommended historical fiction about the Civil War. It tells the story of Rosetta, a strong-willed woman who disguises herself as a man to fight alongside her husband in the war. The novel is rich with historical details and inspired by real accounts of women soldiers in the Civil War. It offers a courageous adventure, a woman\'s search for meaning and individuality, and a poignant story of enduring love.',
 'question': 'Good historical fiction about civil war',
 'retrieved_context': {'context_ids': ['B00EBRUAVC',
   '1561640700',
   '161188263X',
   '006286808X',
   'B01DWTHKFI'],
  'context': ["I Shall Be Near to You: A NovelAn extraordinary novel about a strong-willed woman who disguises herself as a man in order to fight beside her husband, inspired by the letters of a remarkable female soldier who fought in the Civil War. Rosetta doesn't want her new husband Jeremiah to enlist, but he joins up, hoping to make enough money t